# Llama Pipeline Onboarding

Goal: run a small open-weight LLM end-to-end in Colab and understand the tokenize -> forward -> decode pipeline.

Hardware:
- GPU: Tesla T4
- VRAM: 14.56 GB

Model:
- meta-llama/Llama-3.2-1B-Instruct

In [ ]:
import torch

is_cuda_available = torch.cuda.is_available()

if is_cuda_available:
  props = torch.cuda.get_device_properties(0)
  print("GPU: ", torch.cuda.get_device_name(0))
  print("VRAM: ", round(props.total_memory / (1024**3), 2), "GB")

In [ ]:
from huggingface_hub import login

login()

In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto"
)

prompt = "What is a color?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

temperatures = [0.2, 0.7, 1.2]
results = {}

# Optional GPU warmup
_ = model.generate(**inputs, max_new_tokens=5)

for temp in temperatures:
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    output_ids = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=True,
        temperature=temp,
        top_p=0.95
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    input_tokens = inputs["input_ids"].shape[-1]
    total_tokens = output_ids.shape[-1]
    generated_tokens = total_tokens - input_tokens

    tokens_per_second = generated_tokens / elapsed

    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    results[temp] = {
        "text": text,
        "generated_tokens": generated_tokens,
        "elapsed": elapsed,
        "tokens_per_second": tokens_per_second,
    }

print("\n" + "=" * 150)
print(
    f"{'Temp':<8} | {'Gen tokens':<10} | {'Seconds':<10} | {'Tokens/sec':<12} | Output"
)
print("=" * 150)

for temp, result in results.items():
    print(
        f"{temp:<8} | "
        f"{result['generated_tokens']:<10} | "
        f"{result['elapsed']:<10.3f} | "
        f"{result['tokens_per_second']:<12.2f} | "
        f"{result['text']}"
    )
    print("-" * 150)